In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_CURRENT = "sentinel_dev.silver.silver_orders_current"

DIM_CUSTOMER = "sentinel_dev.gold.dim_customer"
DIM_PRODUCT = "sentinel_dev.gold.dim_product"
FACT_ORDERS = "sentinel_dev.gold.fact_orders"
DAILY_SALES = "sentinel_dev.gold.daily_sales"
CONTROL_TABLE = "sentinel_dev.monitoring.gold_watermark"

In [0]:
orders_df = spark.table(SILVER_CURRENT)

print(f"Current valid orders: {orders_df.count():,}")

orders_df.printSchema()

In [0]:
dim_customer_df = (
    orders_df
        .select("customer_id")
        .filter(F.col("customer_id").isNotNull())
        .distinct()

        .withColumn(
            "customer_key",
            F.xxhash64("customer_id")
        )

        .select(
            "customer_key",
            "customer_id"
        )
)

In [0]:
display(dim_customer_df.limit(20))

In [0]:
dim_product_df = (
    orders_df
        .select(
            "product_id",
            "product_name"
        )
        .filter(F.col("product_id").isNotNull())
        .dropDuplicates(["product_id"])

        .withColumn(
            "product_key",
            F.xxhash64("product_id")
        )

        .select(
            "product_key",
            "product_id",
            "product_name"
        )
)

In [0]:
display(dim_product_df.limit(20))

In [0]:
product_conflicts_df = (
    orders_df
        .groupBy("product_id")
        .agg(
            F.countDistinct("product_name").alias(
                "distinct_product_names"
            )
        )
        .filter(
            F.col("distinct_product_names") > 1
        )
)

display(product_conflicts_df)

In [0]:
(
    dim_customer_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_CUSTOMER)
)

(
    dim_product_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_PRODUCT)
)

print("Dimensions written.")

In [0]:
fact_orders_df = (
    orders_df.alias("o")

        .join(
            dim_customer_df.alias("c"),
            F.col("o.customer_id") == F.col("c.customer_id"),
            "left"
        )

        .join(
            dim_product_df.alias("p"),
            F.col("o.product_id") == F.col("p.product_id"),
            "left"
        )

        .select(
            F.col("o.order_id"),

            F.col("c.customer_key"),
            F.col("p.product_key"),

            F.to_date(
                F.col("o.order_timestamp_clean")
            ).alias("order_date"),

            F.col("o.order_timestamp_clean")
                .alias("order_timestamp"),

            F.col("o.quantity"),
            F.col("o.unit_price"),
            F.col("o.total_amount"),

            F.col("o.payment_method"),
            F.col("o.order_status"),
            F.col("o.source_system")
        )
)

In [0]:
display(fact_orders_df.limit(20))

In [0]:
duplicate_facts = (
    fact_orders_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

assert duplicate_facts == 0

print("Fact grain validation passed.")

In [0]:
unresolved_customer_keys = (
    fact_orders_df
        .filter(F.col("customer_key").isNull())
        .count()
)

unresolved_product_keys = (
    fact_orders_df
        .filter(F.col("product_key").isNull())
        .count()
)

print(
    f"""
Unresolved customer keys : {unresolved_customer_keys}
Unresolved product keys  : {unresolved_product_keys}
"""
)

In [0]:
assert fact_orders_df.count() == orders_df.count()

print("Silver → Gold reconciliation passed.")

In [0]:
(
    fact_orders_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FACT_ORDERS)
)

print("fact_orders written.")

In [0]:
daily_sales_df = (
    fact_orders_df

        .filter(
            ~F.col("order_status").isin(
                "CANCELLED"
            )
        )

        .groupBy(
            "order_date"
        )

        .agg(
            F.countDistinct("order_id")
                .alias("orders"),

            F.sum("quantity")
                .alias("units_sold"),

            F.sum("total_amount")
                .alias("gross_revenue"),

            F.avg("total_amount")
                .alias("average_order_value"),

            F.countDistinct("customer_key")
                .alias("unique_customers")
        )

        .orderBy("order_date")
)

In [0]:
display(daily_sales_df)

In [0]:
(
    daily_sales_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DAILY_SALES)
)

In [0]:
if not spark.catalog.tableExists(CONTROL_TABLE):
    (
        spark.createDataFrame(
            [("fact_orders", None)],
            "pipeline_name STRING, last_processed_at TIMESTAMP"
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CONTROL_TABLE)
    )

print("Gold watermark table ready.")

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

print(f"Last processed at: {last_processed_at}")

In [0]:
silver_current_df = spark.table(
    "sentinel_dev.silver.silver_orders_current"
)

In [0]:
if last_processed_at is None:
    changed_orders_df = silver_current_df
else:
    changed_orders_df = (
        silver_current_df
            .filter(
                F.col("ingested_at") > F.lit(last_processed_at)
            )
    )

print(
    f"Changed Silver orders to process: "
    f"{changed_orders_df.count():,}"
)

In [0]:
dim_customer_df = spark.table(
    "sentinel_dev.gold.dim_customer"
)

dim_product_df = spark.table(
    "sentinel_dev.gold.dim_product"
)

In [0]:
gold_changes_df = (
    changed_orders_df.alias("o")

        .join(
            dim_customer_df.alias("c"),
            F.col("o.customer_id") == F.col("c.customer_id"),
            "left"
        )

        .join(
            dim_product_df.alias("p"),
            F.col("o.product_id") == F.col("p.product_id"),
            "left"
        )

        .select(
            F.col("o.order_id"),

            F.col("c.customer_key"),
            F.col("p.product_key"),

            F.to_date(
                F.col("o.order_timestamp_clean")
            ).alias("order_date"),

            F.col("o.order_timestamp_clean")
                .alias("order_timestamp"),

            F.col("o.quantity"),
            F.col("o.unit_price"),
            F.col("o.total_amount"),

            F.col("o.payment_method"),
            F.col("o.order_status"),
            F.col("o.source_system"),

            F.col("o.ingested_at")
        )
)

In [0]:
from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(
    spark,
    "sentinel_dev.gold.fact_orders"
)

(
    gold_delta.alias("target")

    .merge(
        gold_changes_df.alias("source"),
        "target.order_id = source.order_id"
    )

    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

print("Incremental Gold MERGE completed.")

In [0]:
new_watermark = (
    changed_orders_df
        .agg(
            F.max("ingested_at").alias("max_ingested_at")
        )
        .first()["max_ingested_at"]
)

In [0]:
if new_watermark is not None:

    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET last_processed_at = TIMESTAMP '{new_watermark}'
        WHERE pipeline_name = 'fact_orders'
    """)

    print(
        f"Watermark advanced to: {new_watermark}"
    )

In [0]:
display(
    spark.table(CONTROL_TABLE)
)

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

changed_after_run = (
    silver_current_df
        .filter(
            F.col("ingested_at") > F.lit(last_processed_at)
        )
        .count()
)

print(
    f"Rows requiring processing after successful run: "
    f"{changed_after_run}"
)

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

changed_after_run = (
    silver_current_df
        .filter(
            F.col("ingested_at") > F.lit(last_processed_at)
        )
        .count()
)

print(
    f"Rows requiring processing after successful run: "
    f"{changed_after_run}"
)